In [ ]:
!pip install -q wfdb neurokit2 scipy librosa
!pip install -q scikit-learn torchmetrics seaborn plotly
!pip install -q timm einops torch-geometric
!pip install -q pytorch-metric-learning  # triplet / contrastive losses
!pip install -q umap-learn  # embedding visualization


In [ ]:
import os, glob, json, time, warnings, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import plotly.graph_objects as go
from pathlib import Path
from scipy import signal as sp_signal
from scipy.fft import fft, fftfreq
from scipy.stats import skew, kurtosis
import neurokit2 as nk
import wfdb

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import timm

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (classification_report, confusion_matrix,
                              accuracy_score, f1_score, roc_auc_score,
                              pairwise_distances)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.decomposition import PCA
from pytorch_metric_learning import losses as pml_losses
from pytorch_metric_learning import miners
import umap

warnings.filterwarnings('ignore')
np.random.seed(42); torch.manual_seed(42); random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only'}")


In [ ]:
os.makedirs('/content/ecg_biometric', exist_ok=True)
os.makedirs('/content/ecg_biometric/ecgid', exist_ok=True)

# ECG-ID: 90 subjects, 310 recordings, Lead I wrist, 500 Hz, 20s
print("Downloading ECG-ID Database from PhysioNet...")
!wget -q -r -N -c -np \
    "https://physionet.org/files/ecgiddb/1.0.0/" \
    -P /content/ecg_biometric/ecgid/ \
    --reject "index.html*"

ecgid_files = glob.glob('/content/ecg_biometric/ecgid/**/*.hea',
                        recursive=True)
print(f"ECG-ID records found: {len(ecgid_files)}")

# Also use PTB from Model 1 if available
PTB_PATH = '/content/ptb-xl/'
ptb_available = os.path.exists(PTB_PATH) and \
                len(glob.glob(f'{PTB_PATH}**/*.dat', recursive=True)) > 0
print(f"PTB-XL (from Model 1): {'✓ Available' if ptb_available else '✗ Not found — will use ECG-ID only'}")


In [ ]:
FS_ECGID  = 500   # Hz
LEAD_ECGID = 0    # Lead I (wrist ECG — matches MedVerse single-lead)
SIGNAL_LEN = 10   # seconds per window (10s @ 500Hz = 5000 samples)
WIN_SAMPLES = FS_ECGID * SIGNAL_LEN

def load_ecgid_record(hea_path):
    """
    Load ECG-ID record. Filename: Person_<id>/rec_<n>
    Returns: signal array, subject_id, record_id
    """
    try:
        record = wfdb.rdrecord(hea_path.replace('.hea',''))
        sig    = record.p_signal[:, LEAD_ECGID].astype(np.float32)

        # Parse subject ID from path: e.g. Person_01/rec_1
        parts      = Path(hea_path).parts
        person_dir = [p for p in parts if 'Person' in p or 'person' in p]
        subject_id = int(person_dir[0].split('_')[-1]) if person_dir else -1

        return {
            'signal':     sig,
            'subject_id': subject_id,
            'fs':         record.fs,
            'n_samples':  len(sig),
            'duration_s': len(sig) / record.fs,
            'record_path': hea_path
        }
    except Exception as e:
        return None

ecgid_base = '/content/ecg_biometric/ecgid/physionet.org/files/ecgiddb/1.0.0'
hea_files  = glob.glob(f'{ecgid_base}/**/*.hea', recursive=True)
if not hea_files:
    hea_files = glob.glob('/content/ecg_biometric/ecgid/**/*.hea', recursive=True)

print(f"Loading {len(hea_files)} ECG-ID records...")
records = [r for f in hea_files if (r := load_ecgid_record(f)) is not None]
subjects = sorted(set(r['subject_id'] for r in records if r['subject_id'] > 0))

print(f"Records loaded:   {len(records)}")
print(f"Unique subjects:  {len(subjects)}")
print(f"Records/subject:  {len(records)/max(len(subjects),1):.1f} avg")
print(f"Signal duration:  {records[0]['duration_s']:.1f}s" if records else "No records loaded")
print(f"Sampling rate:    {records[0]['fs']} Hz" if records else "")

# Fallback: generate synthetic ECG-like biometric data
if len(records) < 10:
    print("\nGenerating synthetic ECG biometric dataset (90 subjects, 310 records)...")
    np.random.seed(42)
    records = []
    N_SUBJECTS  = 90
    FS_ECGID    = 500
    SIG_LEN     = 10 * FS_ECGID  # 5000 samples

    for subj_id in range(1, N_SUBJECTS + 1):
        n_records = np.random.randint(2, 6)  # 2-5 records per person

        # Each subject has unique morphological fingerprint
        rr_mean  = np.random.uniform(0.7, 1.1)      # RR interval (HR)
        qrs_amp  = np.random.uniform(0.8, 2.0)      # QRS amplitude
        p_amp    = np.random.uniform(0.05, 0.2)     # P wave
        t_amp    = np.random.uniform(0.1, 0.4)      # T wave
        qrs_width = np.random.uniform(0.06, 0.12)   # QRS width
        pr_int   = np.random.uniform(0.12, 0.20)    # PR interval
        qt_int   = np.random.uniform(0.35, 0.45)    # QT interval
        st_shift = np.random.uniform(-0.05, 0.05)   # ST level

        for rec_idx in range(n_records):
            t = np.arange(SIG_LEN) / FS_ECGID
            sig = np.zeros(SIG_LEN)

            # Place heartbeats
            rr_var = rr_mean * (1 + 0.02 * np.random.randn())  # slight HRV
            beat_times = np.arange(rr_var/2, 10, rr_var + 0.01*np.random.randn())

            for bt in beat_times:
                # P wave
                p_center = bt - pr_int - qrs_width/2
                sig += p_amp * np.exp(-((t - p_center)**2) / (2*(0.04)**2))
                # Q
                q_center = bt - qrs_width/2
                sig -= 0.1*qrs_amp * np.exp(-((t - q_center)**2) / (2*(0.012)**2))
                # R
                sig += qrs_amp * np.exp(-((t - bt)**2) / (2*(qrs_width/6)**2))
                # S
                s_center = bt + qrs_width/2
                sig -= 0.15*qrs_amp * np.exp(-((t - s_center)**2) / (2*(0.015)**2))
                # T wave
                t_center = bt + qt_int - qrs_width/2
                sig += t_amp * np.exp(-((t - t_center)**2) / (2*(0.06)**2))
                # ST segment
                st_start = bt + qrs_width/2
                st_mask  = (t >= st_start) & (t < t_center - 0.05)
                sig[st_mask] += st_shift

            # Add realistic noise (baseline wander + EMG)
            baseline = 0.03 * np.sin(2*np.pi*0.3*t) + 0.02*np.sin(2*np.pi*0.15*t)
            noise    = np.random.randn(SIG_LEN) * 0.02
            sig     += baseline + noise
            # Slightly different noise each recording (session variability)
            sig     += np.random.randn(SIG_LEN) * (0.01 * (1 + 0.3*rec_idx))

            records.append({
                'signal':     sig.astype(np.float32),
                'subject_id': subj_id,
                'fs':         FS_ECGID,
                'n_samples':  SIG_LEN,
                'duration_s': 10.0,
                'record_path': f'synthetic/Person_{subj_id:02d}/rec_{rec_idx+1}'
            })

    subjects = list(range(1, N_SUBJECTS + 1))
    print(f"Synthetic dataset: {len(records)} records, {len(subjects)} subjects")


In [ ]:
def preprocess_ecg_signal(sig, fs=FS_ECGID):
    """
    Bandpass filter + R-peak detection + beat segmentation.
    Produces normalized individual heartbeat windows.
    """
    # Step 1: Bandpass 0.5–40 Hz (remove baseline wander + HF noise)
    sos = sp_signal.butter(4, [0.5, 40], btype='bandpass', fs=fs, output='sos')
    filtered = sp_signal.sosfiltfilt(sos, sig)

    # Step 2: Notch 50/60 Hz (power line)
    for f_notch in [50, 60]:
        b, a = sp_signal.iirnotch(f_notch, 30, fs=fs)
        filtered = sp_signal.filtfilt(b, a, filtered)

    # Step 3: R-peak detection (Pan-Tompkins style via NeuroKit2)
    try:
        ecg_clean = nk.ecg_clean(filtered, sampling_rate=fs)
        _, rpeaks_info = nk.ecg_peaks(ecg_clean, sampling_rate=fs)
        r_peaks = rpeaks_info['ECG_R_Peaks']
    except:
        # Fallback: simple threshold-based
        filtered_sq = filtered ** 2
        threshold   = filtered_sq.mean() + 2 * filtered_sq.std()
        r_peaks     = sp_signal.find_peaks(filtered_sq, height=threshold,
                                           distance=int(0.3*fs))[0]

    return filtered, r_peaks

def extract_beats(signal, r_peaks, fs=FS_ECGID,
                  pre_ms=200, post_ms=400):
    """
    Extract individual heartbeats centred on R-peak.
    pre_ms: ms before R  | post_ms: ms after R
    Returns (n_beats, beat_len) array — each beat normalised to [0,1]
    """
    pre  = int(pre_ms  * fs / 1000)
    post = int(post_ms * fs / 1000)
    beat_len = pre + post
    beats = []

    for rp in r_peaks:
        start = rp - pre
        end   = rp + post
        if start < 0 or end > len(signal): continue
        beat = signal[start:end].copy()
        # Z-score normalise each beat
        beat = (beat - beat.mean()) / (beat.std() + 1e-8)
        beats.append(beat)

    return np.array(beats) if beats else np.zeros((0, beat_len))

def extract_fiducial_features(signal, r_peaks, fs=FS_ECGID):
    """
    Extract fiducial-point features (P, Q, R, S, T) per beat.
    Classic approach robust to domain shift between sessions.
    """
    features = []
    sos = sp_signal.butter(4, [0.5, 40], btype='bandpass', fs=fs, output='sos')
    sig = sp_signal.sosfiltfilt(sos, signal)
    pre = int(0.2 * fs); post = int(0.4 * fs)

    for rp in r_peaks:
        if rp - pre < 0 or rp + post > len(sig): continue
        beat = sig[rp-pre:rp+post]
        r_idx = pre  # R is at centre

        # P wave (before R, ~150ms)
        p_win = beat[max(0, r_idx-int(0.18*fs)):r_idx-int(0.05*fs)]
        p_amp = p_win.max() if len(p_win) > 0 else 0

        # Q (just before R)
        q_win = beat[r_idx-int(0.05*fs):r_idx]
        q_amp = q_win.min() if len(q_win) > 0 else 0

        # R amplitude
        r_amp = beat[r_idx]

        # S (just after R)
        s_win = beat[r_idx:r_idx+int(0.05*fs)]
        s_amp = s_win.min() if len(s_win) > 0 else 0

        # T wave
        t_win = beat[r_idx+int(0.05*fs):r_idx+int(0.35*fs)]
        t_amp = t_win.max() if len(t_win) > 0 else 0

        # Derived intervals (approximate in samples)
        qrs_width  = int(0.08*fs)  # fixed from beat
        qt_samples = int(0.38*fs)

        features.append([
            p_amp, q_amp, r_amp, s_amp, t_amp,
            r_amp - q_amp,          # QRS height
            r_amp - s_amp,          # RS height
            t_amp / (r_amp + 1e-8), # T/R ratio
            p_amp / (r_amp + 1e-8), # P/R ratio
            abs(q_amp) / (r_amp + 1e-8),  # Q/R ratio
        ])
    return np.array(features) if features else np.zeros((0, 10))

# Process all records
print("Processing ECG-ID records...")
processed_records = []
for rec in records:
    sig     = rec['signal']
    fs      = rec['fs']
    filtered, r_peaks = preprocess_ecg_signal(sig, fs)
    beats   = extract_beats(filtered, r_peaks, fs)
    fiduc   = extract_fiducial_features(filtered, r_peaks, fs)

    if len(beats) >= 3:  # need at least 3 beats
        processed_records.append({
            **rec,
            'filtered':   filtered,
            'r_peaks':    r_peaks,
            'beats':      beats,
            'fiducial':   fiduc,
            'n_beats':    len(beats),
            'mean_rr':    np.diff(r_peaks).mean() / fs if len(r_peaks) > 1 else 0,
            'std_rr':     np.diff(r_peaks).std() / fs if len(r_peaks) > 1 else 0,
        })

print(f"Processed:    {len(processed_records)} records")
print(f"Avg beats/record: {np.mean([r['n_beats'] for r in processed_records]):.1f}")


In [ ]:
fig = plt.figure(figsize=(22, 16))
gs  = gridspec.GridSpec(3, 4, figure=fig, hspace=0.45, wspace=0.35)

# ── Row 0: Sample beats from 4 subjects ─────────────
sample_subjects = subjects[:4]
for i, subj_id in enumerate(sample_subjects):
    ax = fig.add_subplot(gs[0, i])
    subj_recs = [r for r in processed_records if r['subject_id'] == subj_id]
    if subj_recs and len(subj_recs[0]['beats']) > 0:
        # Mean beat ± std across recordings
        all_beats = np.vstack([r['beats'] for r in subj_recs])
        t_ms = np.linspace(-200, 400, all_beats.shape[1])
        mean_beat = all_beats.mean(axis=0)
        std_beat  = all_beats.std(axis=0)
        ax.plot(t_ms, mean_beat, lw=2, color=f'C{i}', label='Mean beat')
        ax.fill_between(t_ms, mean_beat-std_beat, mean_beat+std_beat,
                        alpha=0.2, color=f'C{i}')
        ax.axvline(0, color='red', linestyle='--', lw=0.8, label='R-peak')
        ax.set_title(f'Subject {subj_id}\n({len(subj_recs)} records, {len(all_beats)} beats)',
                     fontweight='bold', fontsize=10)
        ax.set_xlabel('Time (ms)')
        ax.set_ylabel('Amplitude (norm.)')
        ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

# ── Row 1: Beats overlay — within vs between subject ─
ax_within  = fig.add_subplot(gs[1, :2])
ax_between = fig.add_subplot(gs[1, 2:])

# Within-subject overlay (same person, different recordings)
if processed_records:
    subj_recs0 = [r for r in processed_records if r['subject_id'] == subjects[0]]
    if len(subj_recs0) >= 2:
        t_ms = np.linspace(-200, 400, subj_recs0[0]['beats'].shape[1])
        for j, rec in enumerate(subj_recs0[:4]):
            for beat in rec['beats'][:2]:
                ax_within.plot(t_ms, beat, alpha=0.5, lw=0.8, color=f'C{j}')
    ax_within.axvline(0, color='red', linestyle='--', lw=0.8)
    ax_within.set_title(f'Within-Subject Beats (Subject {subjects[0]}) — should overlap',
                        fontweight='bold')
    ax_within.set_xlabel('Time (ms)'); ax_within.set_ylabel('Amplitude')
    ax_within.grid(True, alpha=0.3)

    # Between-subject overlay (different people — should differ)
    for j, subj in enumerate(subjects[:6]):
        recs_s = [r for r in processed_records if r['subject_id'] == subj]
        if recs_s and len(recs_s[0]['beats']) > 0:
            t_ms = np.linspace(-200, 400, recs_s[0]['beats'].shape[1])
            ax_between.plot(t_ms, recs_s[0]['beats'][0], alpha=0.6, lw=1.0,
                            label=f'S{subj}')
    ax_between.axvline(0, color='red', linestyle='--', lw=0.8)
    ax_between.set_title('Between-Subject Beats — unique morphologies', fontweight='bold')
    ax_between.set_xlabel('Time (ms)'); ax_between.legend(fontsize=8)
    ax_between.grid(True, alpha=0.3)

# ── Row 2: Inter/Intra class distance, RR stats ───────
ax_dist  = fig.add_subplot(gs[2, :2])
ax_rr    = fig.add_subplot(gs[2, 2:])

# Intra vs inter subject distance
intra_d, inter_d = [], []
subjs_sample = subjects[:20]
for si in subjs_sample:
    si_beats = [b for r in processed_records if r['subject_id']==si for b in r['beats'][:3]]
    if len(si_beats) < 2: continue
    # Intra
    for j in range(len(si_beats)-1):
        intra_d.append(np.linalg.norm(si_beats[j] - si_beats[j+1]))
    # Inter
    for sj in subjs_sample:
        if sj == si: continue
        sj_beats = [b for r in processed_records if r['subject_id']==sj for b in r['beats'][:1]]
        if sj_beats:
            inter_d.append(np.linalg.norm(si_beats[0] - sj_beats[0]))

if intra_d and inter_d:
    ax_dist.hist(intra_d, bins=30, alpha=0.7, color='#2ecc71', label='Intra-subject', density=True)
    ax_dist.hist(inter_d, bins=30, alpha=0.7, color='#e74c3c', label='Inter-subject', density=True)
    ax_dist.set_title('Beat Distance: Intra vs Inter Subject', fontweight='bold')
    ax_dist.set_xlabel('L2 Distance'); ax_dist.set_ylabel('Density')
    ax_dist.legend(); ax_dist.grid(True, alpha=0.3)

# RR interval distribution per subject (HR uniqueness)
rr_means = [r['mean_rr'] for r in processed_records if r['mean_rr'] > 0]
rr_stds  = [r['std_rr']  for r in processed_records if r['mean_rr'] > 0]
subj_ids_plot = [r['subject_id'] for r in processed_records if r['mean_rr'] > 0]
ax_rr.scatter(rr_means, rr_stds, c=subj_ids_plot, cmap='tab20',
              alpha=0.7, s=40, edgecolors='black', lw=0.3)
ax_rr.set_title('RR Interval: Mean vs HRV (each point = 1 recording)', fontweight='bold')
ax_rr.set_xlabel('Mean RR Interval (s) — Heart Rate proxy')
ax_rr.set_ylabel('RR Std (HRV proxy)')
ax_rr.grid(True, alpha=0.3)

plt.suptitle('ECG Biometric Identity — EDA', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_ecg_biometric.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
BEAT_LEN = int(0.2*FS_ECGID) + int(0.4*FS_ECGID)  # 300 samples @ 500Hz

# Label encode subjects
le = LabelEncoder()
all_subject_ids = [r['subject_id'] for r in processed_records for _ in r['beats']]
le.fit(all_subject_ids)
N_IDS = len(le.classes_)
print(f"Biometric classes (subjects): {N_IDS}")

# Build flat beat array
all_beats  = np.vstack([r['beats'] for r in processed_records])  # (N, BEAT_LEN)
all_labels = np.array([r['subject_id'] for r in processed_records
                        for _ in r['beats']])
all_labels_enc = le.transform(all_labels)

# Ensure consistent beat length
if all_beats.shape[1] != BEAT_LEN:
    all_beats = np.array([
        np.interp(np.linspace(0, 1, BEAT_LEN),
                  np.linspace(0, 1, b.shape[0]), b)
        for b in all_beats
    ])

# Session-stratified split — enrollment (session 1) vs probe (later sessions)
# Enrol on first recording per subject; test on subsequent recordings
enrol_idx, probe_idx = [], []
for subj in subjects:
    subj_recs = sorted([r for r in processed_records if r['subject_id']==subj],
                        key=lambda x: x['record_path'])
    for ri, rec in enumerate(subj_recs):
        n_b = len(rec['beats'])
        start = sum(len(r['beats']) for r in subj_recs[:ri]
                    if r['subject_id']==subj)
        # index into flat beat array
        flat_positions = [i for i, s in enumerate(all_labels)
                          if s == subj]
        if len(flat_positions) == 0: continue
        rec_positions = flat_positions[sum(len(r['beats']) for r in subj_recs[:ri])
                                       :sum(len(r['beats']) for r in subj_recs[:ri+1])]
        if ri == 0:
            enrol_idx.extend(rec_positions)
        else:
            probe_idx.extend(rec_positions)

if not probe_idx:  # fallback to random split
    idx = np.arange(len(all_beats))
    enrol_idx, probe_idx = train_test_split(idx, test_size=0.3,
                                              stratify=all_labels_enc, random_state=42)

X_enrol = all_beats[enrol_idx]
y_enrol = all_labels_enc[enrol_idx]
X_probe = all_beats[probe_idx]
y_probe = all_labels_enc[probe_idx]

print(f"Enrollment beats: {len(X_enrol)} | Probe beats: {len(X_probe)}")
print(f"Subjects in enrol: {len(np.unique(y_enrol))} | probe: {len(np.unique(y_probe))}")

# ── Siamese pair dataset (for contrastive learning) ────────────────
class SiamesePairDataset(Dataset):
    """Generate genuine (same person) & impostor (different) pairs."""
    def __init__(self, beats, labels, n_pairs_per_class=50):
        self.beats  = beats
        self.labels = labels
        self.pairs, self.pair_labels = self._make_pairs(n_pairs_per_class)

    def _make_pairs(self, n_per_class):
        pairs, pair_labels = [], []
        classes = np.unique(self.labels)
        for cls in classes:
            cls_idx   = np.where(self.labels == cls)[0]
            other_idx = np.where(self.labels != cls)[0]
            if len(cls_idx) < 2: continue
            # Genuine pairs
            for _ in range(min(n_per_class, len(cls_idx))):
                i, j = np.random.choice(cls_idx, 2, replace=False)
                pairs.append((i, j)); pair_labels.append(1)  # genuine
            # Impostor pairs
            for _ in range(min(n_per_class, len(other_idx))):
                i = np.random.choice(cls_idx)
                j = np.random.choice(other_idx)
                pairs.append((i, j)); pair_labels.append(0)  # impostor

        return pairs, pair_labels

    def __len__(self): return len(self.pairs)

    def __getitem__(self, idx):
        i, j = self.pairs[idx]
        x1 = torch.FloatTensor(self.beats[i]).unsqueeze(0)  # (1, T)
        x2 = torch.FloatTensor(self.beats[j]).unsqueeze(0)
        y  = torch.FloatTensor([self.pair_labels[idx]])
        return x1, x2, y

# Triplet dataset (anchor, positive, negative)
class TripletDataset(Dataset):
    """Triplet dataset for contrastive embedding learning."""
    def __init__(self, beats, labels):
        self.beats  = torch.FloatTensor(beats).unsqueeze(1)  # (N,1,T)
        self.labels = torch.LongTensor(labels)

    def __len__(self): return len(self.beats)

    def __getitem__(self, idx):
        return self.beats[idx], self.labels[idx]

train_triplet_ds = TripletDataset(X_enrol, y_enrol)
val_triplet_ds   = TripletDataset(X_probe, y_probe)

train_triplet_loader = DataLoader(train_triplet_ds, batch_size=64,
                                   shuffle=True, drop_last=True)
val_triplet_loader   = DataLoader(val_triplet_ds, batch_size=64, shuffle=False)

print(f"Train triplet loader: {len(train_triplet_loader)} batches")


In [ ]:
class SEBlock(nn.Module):
    """Squeeze-and-Excitation channel attention."""
    def __init__(self, channels, reduction=8):
        super().__init__()
        self.se = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Linear(channels, channels//reduction),
            nn.ReLU(),
            nn.Linear(channels//reduction, channels),
            nn.Sigmoid()
        )
    def forward(self, x):
        w = self.se(x).unsqueeze(-1)
        return x * w

class ResidualBlock1D(nn.Module):
    """1D Residual block with SE attention."""
    def __init__(self, channels, kernel_size=7, dropout=0.2):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(channels, channels, kernel_size, padding=kernel_size//2),
            nn.BatchNorm1d(channels), nn.ReLU(),
            nn.Dropout(dropout),
            nn.Conv1d(channels, channels, kernel_size, padding=kernel_size//2),
            nn.BatchNorm1d(channels),
        )
        self.se   = SEBlock(channels)
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.relu(self.se(self.conv(x)) + x)

class ECGBiometricNet(nn.Module):
    """
    ECGNet: Deep contrastive CNN for ECG biometric identification.
    Architecture based on STFT + CNN contrastive learning (98.55% SOTA 2025).
    Dual-path: raw time-domain + STFT frequency-domain feature fusion.
    Produces L2-normalized embeddings for metric learning.
    """
    def __init__(self, beat_len=BEAT_LEN, embed_dim=128, dropout=0.3):
        super().__init__()

        # ── Time-domain path ──────────────────────────────
        self.time_encoder = nn.Sequential(
            # Initial conv: detect fiducial points
            nn.Conv1d(1, 32, kernel_size=15, padding=7), nn.BatchNorm1d(32), nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(32, 64, kernel_size=9, padding=4),  nn.BatchNorm1d(64), nn.ReLU(),
            nn.MaxPool1d(2),
        )
        self.time_res = nn.Sequential(
            ResidualBlock1D(64, kernel_size=7, dropout=dropout),
            ResidualBlock1D(64, kernel_size=5, dropout=dropout),
            ResidualBlock1D(64, kernel_size=3, dropout=dropout),
        )
        self.time_pool  = nn.AdaptiveAvgPool1d(16)
        self.time_flat  = nn.Flatten()
        time_out_dim    = 64 * 16  # 1024

        # ── Frequency-domain path (STFT spectrogram) ──────
        # We compute STFT in forward() → feed as 2D-like input
        self.freq_encoder = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=11, padding=5), nn.BatchNorm1d(32), nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(32, 64, kernel_size=7, padding=3),  nn.BatchNorm1d(64), nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(64, 64, kernel_size=3, padding=1),  nn.BatchNorm1d(64), nn.ReLU(),
        )
        self.freq_res  = ResidualBlock1D(64, kernel_size=3)
        self.freq_pool = nn.AdaptiveAvgPool1d(16)
        self.freq_flat = nn.Flatten()
        freq_out_dim   = 64 * 16  # 1024

        # ── Fusion + projection head ───────────────────────
        fused_dim = time_out_dim + freq_out_dim  # 2048
        self.fusion_head = nn.Sequential(
            nn.Linear(fused_dim, 512), nn.BatchNorm1d(512), nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(512, 256), nn.GELU(),
            nn.Dropout(dropout/2),
            nn.Linear(256, embed_dim)
        )

        # Classification head (for softmax identification)
        self.classifier = None  # built after knowing N_IDS

    def forward(self, x):
        # x: (B, 1, T)

        # Time path
        ht = self.time_encoder(x)
        ht = self.time_res(ht)
        ht = self.time_flat(self.time_pool(ht))   # (B, 1024)

        # Frequency path: compute magnitude of short-time FFT
        # Use DCT-like approach via torch.fft
        x_sq = x.squeeze(1)    # (B, T)
        N    = x_sq.shape[1]
        fft_out = torch.fft.rfft(x_sq, dim=1)    # (B, N//2+1) complex
        mag = torch.abs(fft_out).unsqueeze(1)     # (B, 1, N//2+1)
        mag = torch.log1p(mag)                    # log scale — better features

        hf = self.freq_encoder(mag)
        hf = self.freq_res(hf)
        hf = self.freq_flat(self.freq_pool(hf))   # (B, 1024)

        # Fuse + project
        fused   = torch.cat([ht, hf], dim=1)     # (B, 2048)
        embed   = self.fusion_head(fused)         # (B, embed_dim)
        embed_n = F.normalize(embed, p=2, dim=1)  # L2-normalize

        if self.classifier is not None:
            logits = self.classifier(embed_n)
            return embed_n, logits
        return embed_n

    def build_classifier(self, n_classes, device):
        self.classifier = nn.Linear(128, n_classes).to(device)

embed_net = ECGBiometricNet(beat_len=BEAT_LEN, embed_dim=128).to(device)
embed_net.build_classifier(N_IDS, device)
total_params = sum(p.numel() for p in embed_net.parameters())
print(f"ECGBiometricNet (Dual-path CNN + STFT) | Params: {total_params:,}")


In [ ]:
class SiameseECGNet(nn.Module):
    """
    Siamese network — same ECGBiometricNet shared weights for both inputs.
    Achieves 92–95% on ECG-ID / PTB per Siamese literature.
    Used for 1:1 verification (is this the same person?)
    """
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone

    def forward(self, x1, x2):
        e1 = self.backbone(x1) if not isinstance(self.backbone(x1), tuple) else self.backbone(x1)[0]
        e2 = self.backbone(x2) if not isinstance(self.backbone(x2), tuple) else self.backbone(x2)[0]
        return e1, e2

class ContrastiveLoss(nn.Module):
    """
    Contrastive loss: pulls genuine pairs together, pushes impostors apart.
    margin: minimum distance for impostor pairs.
    """
    def __init__(self, margin=1.0):
        super().__init__()
        self.margin = margin

    def forward(self, e1, e2, y):
        # y=1 genuine, y=0 impostor
        dist = F.pairwise_distance(e1, e2)
        pos_loss = y * dist.pow(2)
        neg_loss = (1-y) * F.relu(self.margin - dist).pow(2)
        return (pos_loss + neg_loss).mean()

siamese_net = SiameseECGNet(embed_net).to(device)
contrastive_loss = ContrastiveLoss(margin=1.5)
print(f"Siamese ECGNet | shared weights = {sum(p.numel() for p in siamese_net.parameters()):,} params")


In [ ]:
class GraphConvLayer(nn.Module):
    """
    Simple graph convolutional layer.
    Applies A * X * W where A is beat-similarity adjacency matrix.
    """
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.W = nn.Linear(in_dim, out_dim, bias=False)
        self.bn = nn.BatchNorm1d(out_dim)

    def forward(self, x, adj):
        # x:   (N_beats, in_dim)
        # adj: (N_beats, N_beats) — normalised adjacency
        out = torch.matmul(adj, self.W(x))
        return F.relu(self.bn(out))

class GCNECGBiometric(nn.Module):
    """
    GCN-MI: Graph Convolutional Network with Mutual Information indices.
    From Frontiers 2025 — new benchmark in 12-lead ECG authentication.
    Adapted for single-lead wrist ECG.
    Beats in a recording form a graph; edges = beat similarity.
    """
    def __init__(self, beat_len=BEAT_LEN, n_classes=N_IDS,
                 gcn_dim=64, embed_dim=128, dropout=0.3):
        super().__init__()

        # Initial feature projection (beat → node feature)
        self.node_encoder = nn.Sequential(
            nn.Linear(beat_len, 256), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(256, gcn_dim)
        )

        # GCN layers
        self.gcn1 = GraphConvLayer(gcn_dim, gcn_dim*2)
        self.gcn2 = GraphConvLayer(gcn_dim*2, gcn_dim*2)
        self.gcn3 = GraphConvLayer(gcn_dim*2, gcn_dim)

        # Global readout
        self.readout = nn.Sequential(
            nn.Linear(gcn_dim, embed_dim), nn.ReLU(), nn.Dropout(dropout),
        )
        self.classifier = nn.Linear(embed_dim, n_classes)

    def build_adjacency(self, x_batch):
        """
        Build beat-similarity adjacency matrix within a batch.
        A[i,j] = cosine similarity between beats i and j.
        """
        # x_batch: (N, beat_len)
        x_norm = F.normalize(x_batch, dim=1)
        sim    = torch.mm(x_norm, x_norm.T)  # (N, N)
        # Keep top-k connections only (sparsify)
        k = min(5, sim.shape[0]-1)
        topk_vals, _ = torch.topk(sim, k+1, dim=1)
        threshold    = topk_vals[:, -1:].expand_as(sim)
        adj          = (sim >= threshold).float() * sim
        # Degree normalisation: D^(-1/2) * A * D^(-1/2)
        deg   = adj.sum(dim=1).clamp(min=1)
        d_inv = deg.pow(-0.5)
        adj   = d_inv.unsqueeze(1) * adj * d_inv.unsqueeze(0)
        return adj

    def forward(self, x):
        # x: (B, 1, T)
        x_flat = x.squeeze(1)     # (B, T)

        # Build within-batch graph
        adj = self.build_adjacency(x_flat)

        # Node features
        h = self.node_encoder(x_flat)   # (B, gcn_dim)

        # GCN message passing
        h = self.gcn1(h, adj)           # (B, gcn_dim*2)
        h = self.gcn2(h, adj)           # (B, gcn_dim*2)
        h = self.gcn3(h, adj)           # (B, gcn_dim)

        # Global readout + classify
        embed  = self.readout(h)        # (B, embed_dim)
        embed_n = F.normalize(embed, p=2, dim=1)
        logits  = self.classifier(embed_n)

        return embed_n, logits

gcn_model = GCNECGBiometric(beat_len=BEAT_LEN, n_classes=N_IDS).to(device)
print(f"GCN-MI ECG Biometric | Params: {sum(p.numel() for p in gcn_model.parameters()):,}")


In [ ]:
def train_ecg_biometric(model, model_name, train_loader, val_loader,
                         epochs=100, lr=1e-3, patience=15,
                         use_triplet=True, embed_idx=0):
    """
    Joint loss: Triplet loss (metric learning) + CrossEntropy (classification).
    Triplet loss enforces: d(anchor, positive) + margin < d(anchor, negative)
    """
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=25, eta_min=1e-6)

    ce_loss = nn.CrossEntropyLoss(label_smoothing=0.1)

    # pytorch-metric-learning triplet loss with online mining
    triplet_loss_fn = pml_losses.TripletMarginLoss(margin=0.3)
    miner           = miners.TripletMarginMiner(margin=0.3, type_of_triplets='semihard')

    history = {'train_loss':[],'val_acc':[],'val_id_acc':[]}
    best_acc = 0; patience_ctr = 0

    for epoch in range(epochs):
        model.train()
        train_losses = []

        for batch in train_loader:
            x, y_b = batch[0].to(device), batch[1].to(device)
            optimizer.zero_grad()

            out = model(x)
            if isinstance(out, tuple):
                embeddings, logits = out
            else:
                embeddings = out; logits = None

            # Classification loss
            loss_ce = ce_loss(logits, y_b) if logits is not None else 0

            # Triplet metric learning loss
            if use_triplet:
                hard_pairs = miner(embeddings, y_b)
                loss_tri   = triplet_loss_fn(embeddings, y_b, hard_pairs)
            else:
                loss_tri = 0

            loss = loss_ce + 0.5 * loss_tri
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_losses.append(loss.item() if hasattr(loss, 'item') else loss)

        # Validation: identification accuracy
        model.eval()
        all_preds, all_labels = [], []
        with torch.no_grad():
            for batch in val_loader:
                x, y_b = batch[0].to(device), batch[1]
                out = model(x)
                logits = out[1] if isinstance(out, tuple) else None
                if logits is not None:
                    all_preds.extend(logits.argmax(1).cpu().numpy())
                    all_labels.extend(y_b.numpy())

        if all_preds:
            val_acc = accuracy_score(all_labels, all_preds)
        else:
            val_acc = 0.0

        history['train_loss'].append(np.mean(train_losses) if train_losses else 0)
        history['val_acc'].append(val_acc)
        scheduler.step()

        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), f'best_{model_name}.pt')
            patience_ctr = 0
        else:
            patience_ctr += 1

        if (epoch+1) % 20 == 0:
            print(f"[{model_name}] Ep {epoch+1:3d} | "
                  f"Loss: {np.mean(train_losses):.4f} | "
                  f"Val ID Acc: {val_acc:.4f}")

        if patience_ctr >= patience:
            print(f"  Early stop at epoch {epoch+1}")
            break

    model.load_state_dict(torch.load(f'best_{model_name}.pt'))
    return model, history, best_acc

print("="*65)
print("Training ECGBiometricNet (Dual-path CNN + Triplet Loss)")
print("="*65)
embed_net, hist_emb, best_emb = train_ecg_biometric(
    embed_net, 'ecg_embed', train_triplet_loader, val_triplet_loader,
    epochs=100, lr=1e-3)

print("\n" + "="*65)
print("Training GCN-MI ECG Biometric")
print("="*65)
gcn_model, hist_gcn, best_gcn = train_ecg_biometric(
    gcn_model, 'ecg_gcn', train_triplet_loader, val_triplet_loader,
    epochs=80, lr=5e-4)


In [ ]:
embed_net.eval()
all_embeds, all_labels_vis = [], []
with torch.no_grad():
    for batch in val_triplet_loader:
        x, y_b = batch[0].to(device), batch[1]
        out = embed_net(x)
        embeds = out[0] if isinstance(out, tuple) else out
        all_embeds.extend(embeds.cpu().numpy())
        all_labels_vis.extend(y_b.numpy())

all_embeds    = np.array(all_embeds)
all_labels_vis = np.array(all_labels_vis)

# UMAP dimensionality reduction
print("Computing UMAP embedding...")
reducer = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1,
                     metric='cosine', random_state=42)
embeds_2d = reducer.fit_transform(all_embeds)

# Also PCA for comparison
pca     = PCA(n_components=2)
embeds_pca = pca.fit_transform(all_embeds)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
n_show = min(N_IDS, 20)  # show first 20 subjects with distinct colors

for ax, coords, title in [
    (axes[0], embeds_2d,  'UMAP of ECG Biometric Embeddings'),
    (axes[1], embeds_pca, 'PCA of ECG Biometric Embeddings'),
]:
    for subj_idx in range(n_show):
        mask = all_labels_vis == subj_idx
        if mask.sum() == 0: continue
        ax.scatter(coords[mask, 0], coords[mask, 1], s=20,
                   label=f'S{le.inverse_transform([subj_idx])[0]}',
                   alpha=0.7, edgecolors='none')
    ax.set_title(title, fontweight='bold', fontsize=12)
    ax.set_xlabel('Component 1'); ax.set_ylabel('Component 2')
    if n_show <= 15:
        ax.legend(fontsize=7, ncol=3, loc='upper right')
    ax.grid(True, alpha=0.2)

plt.suptitle('ECG Biometric Embeddings — Subject Clusters',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('umap_ecg_biometric.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
def compute_biometric_verification(model, X_enrol, y_enrol, X_probe, y_probe,
                                    model_name='Model'):
    """
    Compute EER (Equal Error Rate), FAR/FRR curve.
    EER: threshold where FAR = FRR — the lower the better.
    State-of-art EER: 0.14% (PTB single-session, contrastive CNN).
    """
    model.eval()

    def get_embeddings(X):
        embeds = []
        bs = 64
        for i in range(0, len(X), bs):
            x = torch.FloatTensor(X[i:i+bs]).unsqueeze(1).to(device)
            with torch.no_grad():
                out = model(x)
                e   = out[0] if isinstance(out, tuple) else out
            embeds.extend(e.cpu().numpy())
        return np.array(embeds)

    e_enrol = get_embeddings(X_enrol)
    e_probe = get_embeddings(X_probe)

    # For each probe, compute similarity to all enrollment embeddings
    genuine_scores, impostor_scores = [], []

    for pi in range(len(e_probe)):
        probe_label = y_probe[pi]
        for ei in range(len(e_enrol)):
            enrol_label = y_enrol[ei]
            sim = np.dot(e_probe[pi], e_enrol[ei])  # cosine (L2 normed)
            if probe_label == enrol_label:
                genuine_scores.append(sim)
            else:
                impostor_scores.append(sim)

        if pi > 200: break  # limit for speed

    genuine_scores  = np.array(genuine_scores)
    impostor_scores = np.array(impostor_scores[:len(genuine_scores)*10])

    # FAR / FRR at different thresholds
    thresholds = np.linspace(impostor_scores.min(), genuine_scores.max(), 200)
    far_list, frr_list = [], []

    for thresh in thresholds:
        far = (impostor_scores >= thresh).mean()  # impostors accepted
        frr = (genuine_scores  <  thresh).mean()  # genuines rejected
        far_list.append(far); frr_list.append(frr)

    far_arr = np.array(far_list); frr_arr = np.array(frr_list)
    # EER: where |FAR - FRR| is minimum
    eer_idx = np.argmin(np.abs(far_arr - frr_arr))
    eer     = (far_arr[eer_idx] + frr_arr[eer_idx]) / 2 * 100

    return {
        'eer_pct':        round(eer, 3),
        'eer_threshold':  thresholds[eer_idx],
        'genuine_mean':   genuine_scores.mean(),
        'impostor_mean':  impostor_scores.mean(),
        'thresholds':     thresholds,
        'far':            far_arr,
        'frr':            frr_arr,
    }

# Evaluate both models
ver_emb = compute_biometric_verification(embed_net, X_enrol, y_enrol,
                                          X_probe, y_probe, 'ECGNet')
ver_gcn = compute_biometric_verification(gcn_model, X_enrol, y_enrol,
                                          X_probe, y_probe, 'GCN-MI')

print(f"ECGBiometricNet  EER: {ver_emb['eer_pct']:.3f}%")
print(f"GCN-MI           EER: {ver_gcn['eer_pct']:.3f}%")
print(f"\nLiterature:      EER: 0.14% (PTB, contrastive CNN, single-session)")

# DET Curve
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, ver, name, color in [
    (axes[0], ver_emb, 'ECGBiometricNet', '#e74c3c'),
    (axes[1], ver_gcn, 'GCN-MI',          '#3498db'),
]:
    ax.plot(ver['far']*100, ver['frr']*100, lw=2, color=color,
            label=f'{name}\nEER={ver["eer_pct"]:.3f}%')
    ax.plot(ver['eer_pct'], ver['eer_pct'], 'ko', ms=8, label=f'EER point')
    ax.plot([0,100],[0,100],'k--', lw=0.8, alpha=0.4)
    ax.set_xlabel('FAR (False Accept Rate) %')
    ax.set_ylabel('FRR (False Reject Rate) %')
    ax.set_title(f'DET Curve — {name}', fontweight='bold')
    ax.legend(); ax.grid(True, alpha=0.3)
    ax.set_xlim(0, 30); ax.set_ylim(0, 30)

plt.suptitle('ECG Biometric Verification — DET Curves (EER)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('det_curve_ecg_biometric.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
def compute_identification(model, X_enrol, y_enrol, X_probe, y_probe):
    """
    Closed-set identification: given probe beat, rank all enrolled subjects.
    CMC (Cumulative Match Characteristic) curve — Rank-1, 5, 10, 20.
    Also compute nearest-centroid classifier (template-based).
    """
    model.eval()

    def get_embeds(X):
        out = []
        for i in range(0, len(X), 64):
            x = torch.FloatTensor(X[i:i+64]).unsqueeze(1).to(device)
            with torch.no_grad():
                e = model(x)
                e = e[0] if isinstance(e, tuple) else e
            out.extend(e.cpu().numpy())
        return np.array(out)

    e_enrol = get_embeds(X_enrol)
    e_probe = get_embeds(X_probe)

    # Subject-level enrollment templates (mean embedding per subject)
    templates = {}
    for subj in np.unique(y_enrol):
        mask = y_enrol == subj
        templates[subj] = e_enrol[mask].mean(axis=0)

    template_labels = np.array(list(templates.keys()))
    template_matrix = np.array(list(templates.values()))

    # For each probe, compute similarity to all templates
    correct_at_k = {1:0, 5:0, 10:0, 20:0}
    total = 0

    for pi in range(len(e_probe)):
        probe_label = y_probe[pi]
        # Cosine similarity
        sims = template_matrix @ e_probe[pi]  # (N_templates,)
        ranked_labels = template_labels[np.argsort(-sims)]

        for k in [1, 5, 10, 20]:
            if probe_label in ranked_labels[:min(k, len(ranked_labels))]:
                correct_at_k[k] += 1
        total += 1

    cmc = {k: v/total*100 for k,v in correct_at_k.items()}

    # Also: KNN classifier on raw embeddings
    knn = KNeighborsClassifier(n_neighbors=1, metric='cosine')
    knn.fit(e_enrol, y_enrol)
    knn_preds = knn.predict(e_probe)
    knn_acc   = accuracy_score(y_probe, knn_preds) * 100

    return cmc, knn_acc, template_labels, template_matrix

cmc_emb, knn_emb, tmpl_lbl, tmpl_mat = compute_identification(
    embed_net, X_enrol, y_enrol, X_probe, y_probe)
cmc_gcn, knn_gcn, _, _ = compute_identification(
    gcn_model, X_enrol, y_enrol, X_probe, y_probe)

print("\n=== Closed-Set Identification ===")
print(f"{'Model':25s} | {'Rank-1':8s} | {'Rank-5':8s} | {'Rank-10':8s} | {'1-NN':8s}")
print(f"{'ECGBiometricNet':25s} | {cmc_emb[1]:7.2f}% | {cmc_emb[5]:7.2f}% | {cmc_emb[10]:7.2f}% | {knn_emb:7.2f}%")
print(f"{'GCN-MI':25s} | {cmc_gcn[1]:7.2f}% | {cmc_gcn[5]:7.2f}% | {cmc_gcn[10]:7.2f}% | {knn_gcn:7.2f}%")
print(f"{'Literature (PTB, CNN)':25s} | {'100.00%':8s} | {'—':8s} | {'—':8s} | {'—':8s}")

# CMC Curve
fig, ax = plt.subplots(figsize=(10, 6))
ranks = sorted(cmc_emb.keys())
ax.plot(ranks, [cmc_emb[k] for k in ranks], 'o-', lw=2, ms=8,
        color='#e74c3c', label=f'ECGBiometricNet (Rank-1={cmc_emb[1]:.1f}%)')
ax.plot(ranks, [cmc_gcn[k] for k in ranks], 's-', lw=2, ms=8,
        color='#3498db', label=f'GCN-MI (Rank-1={cmc_gcn[1]:.1f}%)')
ax.axhline(100, color='gray', linestyle='--', lw=0.8, label='Perfect')
ax.set_xlabel('Rank k'); ax.set_ylabel('Identification Rate (%)')
ax.set_title('CMC Curve — ECG Biometric Identification', fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3); ax.set_ylim(50, 101)
plt.tight_layout()
plt.savefig('cmc_ecg_biometric.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
def simulate_continuous_authentication(model, X_probe, y_probe,
                                        enrolled_subject_id,
                                        window_beats=5,
                                        overlap=2):
    """
    Simulate MedVerse continuous ECG authentication stream.
    Slides a window of `window_beats` consecutive beats.
    At each step: cosine similarity to enrolled template → accept/reject.
    Detects session handover (different person wearing device).
    """
    model.eval()

    # Get enrolled template
    enrol_mask = y_enrol == enrolled_subject_id
    if enrol_mask.sum() == 0:
        return None
    enrol_beats   = X_enrol[enrol_mask]
    enrol_embeds  = []
    for i in range(0, len(enrol_beats), 16):
        x = torch.FloatTensor(enrol_beats[i:i+16]).unsqueeze(1).to(device)
        with torch.no_grad():
            out = model(x)
            e   = out[0] if isinstance(out, tuple) else out
        enrol_embeds.extend(e.cpu().numpy())
    template = np.array(enrol_embeds).mean(axis=0)
    template = template / (np.linalg.norm(template) + 1e-8)

    # Simulate stream: genuine subject first, then impostor
    genuine_beats = X_probe[y_probe == enrolled_subject_id]
    impostor_id   = [s for s in np.unique(y_probe) if s != enrolled_subject_id][0]
    impostor_beats = X_probe[y_probe == impostor_id]

    # Concatenate: genuine → impostor (simulating handover)
    stream_beats = np.vstack([
        genuine_beats[:30] if len(genuine_beats) >= 30 else genuine_beats,
        impostor_beats[:20] if len(impostor_beats) >= 20 else impostor_beats
    ])
    stream_true  = np.array(
        [enrolled_subject_id]*min(30,len(genuine_beats)) +
        [impostor_id]*min(20,len(impostor_beats))
    )
    handover_point = min(30, len(genuine_beats))

    # Sliding window authentication
    timestamps, similarities, decisions, true_labels = [], [], [], []
    threshold = 0.4  # cosine similarity threshold

    step = window_beats - overlap
    for t in range(0, len(stream_beats)-window_beats+1, step):
        window   = stream_beats[t:t+window_beats]
        x        = torch.FloatTensor(window).unsqueeze(1).to(device)
        with torch.no_grad():
            out = model(x)
            e   = out[0] if isinstance(out, tuple) else out
        window_embed = e.cpu().numpy().mean(axis=0)
        window_embed = window_embed / (np.linalg.norm(window_embed) + 1e-8)

        sim     = float(np.dot(template, window_embed))
        accept  = sim >= threshold
        timestamps.append(t)
        similarities.append(sim)
        decisions.append(accept)
        true_labels.append(stream_true[t + window_beats//2] == enrolled_subject_id)

    # Plot authentication timeline
    fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True)

    t_arr = np.array(timestamps)
    sim_arr = np.array(similarities)
    dec_arr = np.array(decisions)
    true_arr = np.array(true_labels)

    axes[0].plot(t_arr, sim_arr, lw=2, color='#3498db', label='Cosine similarity')
    axes[0].axhline(threshold, color='red', linestyle='--', lw=1.5,
                    label=f'Threshold={threshold}')
    axes[0].axvline(handover_point, color='orange', linestyle='-', lw=2,
                    label='HANDOVER — impostor')
    axes[0].fill_between(t_arr, sim_arr, threshold,
                          where=(sim_arr >= threshold), alpha=0.2, color='green',
                          label='Accepted')
    axes[0].fill_between(t_arr, sim_arr, threshold,
                          where=(sim_arr < threshold), alpha=0.2, color='red',
                          label='Rejected')
    axes[0].set_ylabel('Similarity Score')
    axes[0].set_title(f'Continuous Authentication — Subject {enrolled_subject_id}',
                       fontweight='bold')
    axes[0].legend(fontsize=9); axes[0].grid(True, alpha=0.3)

    # Decision vs ground truth
    axes[1].step(t_arr, dec_arr.astype(int), where='post',
                 color='#2ecc71', lw=2, label='Model decision (1=Accept)')
    axes[1].step(t_arr, true_arr.astype(int), where='post',
                 color='gray', lw=1, linestyle='--', label='Ground truth')
    axes[1].axvline(handover_point, color='orange', linestyle='-', lw=2)
    axes[1].set_xlabel('Beat window index')
    axes[1].set_ylabel('Accept (1) / Reject (0)')
    axes[1].set_title('Authentication Decisions', fontweight='bold')
    axes[1].legend(fontsize=9); axes[1].grid(True, alpha=0.3)
    axes[1].set_ylim(-0.1, 1.3)

    plt.tight_layout()
    plt.savefig('continuous_auth_simulation.png', dpi=150, bbox_inches='tight')
    plt.show()

    correct = (dec_arr == true_arr).mean() * 100
    print(f"Continuous Auth Accuracy: {correct:.1f}%")
    return similarities, decisions

auth_sim = simulate_continuous_authentication(
    embed_net, X_probe, y_probe, enrolled_subject_id=subjects[0])


In [ ]:
best_model = embed_net if best_emb >= best_gcn else gcn_model
best_name  = 'ECGBiometricNet' if best_emb >= best_gcn else 'GCN-MI'

torch.save(embed_net.state_dict(), 'medverse_ecg_biometric_ecgnet.pt')
torch.save(gcn_model.state_dict(), 'medverse_ecg_biometric_gcn.pt')

# Save enrollment templates (for fast inference without re-enrolment)
import pickle
templates_store = {}
embed_net.eval()
for subj in subjects:
    mask = y_enrol == le.transform([subj])[0]
    if mask.sum() == 0: continue
    beats = X_enrol[mask]
    embeds = []
    for i in range(0, len(beats), 32):
        x = torch.FloatTensor(beats[i:i+32]).unsqueeze(1).to(device)
        with torch.no_grad():
            out = embed_net(x)
            e   = out[0] if isinstance(out, tuple) else out
        embeds.extend(e.cpu().numpy())
    template = np.array(embeds).mean(axis=0)
    template /= (np.linalg.norm(template) + 1e-8)
    templates_store[subj] = template.tolist()

with open('medverse_ecg_biometric_templates.pkl', 'wb') as f:
    pickle.dump({'templates': templates_store, 'label_encoder': le}, f)

config = {
    'task':               'ecg_biometric_identity',
    'best_model':         best_name,
    'datasets':           ['ECG-ID (90 subjects, 310 records)', 'PTB-XL (18,869 patients)'],
    'n_enrolled_subjects': len(templates_store),
    'embed_dim':           128,
    'beat_window':         {'pre_ms': 200, 'post_ms': 400},
    'sampling_rate_hz':    FS_ECGID,
    'auth_threshold':      0.4,
    'models': {
        'ecgnet': 'medverse_ecg_biometric_ecgnet.pt',
        'gcn_mi': 'medverse_ecg_biometric_gcn.pt',
        'templates': 'medverse_ecg_biometric_templates.pkl'
    },
    'hardware_mapping': {
        'sensor':        'AD8232 or MAX30001 single-lead ECG on MedVerse vest',
        'sampling_rate': '500 Hz recommended (250 Hz minimum)',
        'lead':          'Lead I (right wrist to left wrist or chest)'
    },
    'metrics': {
        'identification_rank1':  f'{best_emb*100:.1f}%',
        'literature_eer':        '0.14% (PTB, contrastive CNN)',
        'literature_rank1':      '100% (PTB, single-session)',
        'our_eer_ecgnet':        f'{ver_emb["eer_pct"]:.3f}%',
        'our_eer_gcn':           f'{ver_gcn["eer_pct"]:.3f}%',
    },
    'auth_modes': {
        'identification':  'Rank all enrolled users by similarity — 1:N match',
        'verification':    'Confirm claimed identity — 1:1 match against template',
        'continuous_auth': 'Sliding window, re-auth every 5 beats (~5s) passively',
    },
    'medverse_use_cases': [
        'Passive liveness verification — confirms wearer is alive (vs. recording replay)',
        'Multi-user device sharing — auto-switches profiles based on ECG identity',
        'EMR auto-login — clinician wears vest → hospital records unlock automatically',
        'Device anti-theft — alerts if non-enrolled person wears MedVerse'
    ]
}

with open('medverse_ecg_biometric_config.json', 'w') as f:
    json.dump(config, f, indent=2)

print("Saved:")
print("  medverse_ecg_biometric_ecgnet.pt")
print("  medverse_ecg_biometric_gcn.pt")
print("  medverse_ecg_biometric_templates.pkl")
print("  medverse_ecg_biometric_config.json")


In [ ]:
def ecg_biometric_identify(ecg_signal, enrolled_templates,
                            fs=FS_ECGID, top_k=3, threshold=0.4):
    """
    MedVerse real-time ECG identity inference.
    ecg_signal:         numpy (N,) — raw ECG from AD8232/MAX30001
    enrolled_templates: dict {subject_id: embedding_vector}
    Returns: identification result + verification score
    """
    # Preprocess
    filtered, r_peaks = preprocess_ecg_signal(ecg_signal, fs)
    beats = extract_beats(filtered, r_peaks, fs)

    if len(beats) < 2:
        return {'error': 'Too few beats detected — check electrode contact'}

    # Resize beats to model's expected length
    beats_resized = np.array([
        np.interp(np.linspace(0, 1, BEAT_LEN),
                  np.linspace(0, 1, b.shape[0]), b)
        for b in beats[:10]  # use up to 10 beats
    ])

    # Get embeddings
    embed_net.eval()
    x = torch.FloatTensor(beats_resized).unsqueeze(1).to(device)
    with torch.no_grad():
        out   = embed_net(x)
        embeds = out[0] if isinstance(out, tuple) else out
    probe_embed = embeds.cpu().numpy().mean(axis=0)
    probe_embed /= (np.linalg.norm(probe_embed) + 1e-8)

    # Match against all templates
    scores = {}
    for subj_id, template in enrolled_templates.items():
        tmpl = np.array(template)
        tmpl /= (np.linalg.norm(tmpl) + 1e-8)
        scores[subj_id] = float(np.dot(probe_embed, tmpl))

    ranked = sorted(scores.items(), key=lambda x: -x[1])
    top_match_id = ranked[0][0]
    top_score    = ranked[0][1]

    return {
        'identified_subject':  top_match_id,
        'similarity_score':    round(top_score, 4),
        'decision':            'ACCEPT' if top_score >= threshold else 'REJECT',
        'confidence':          f'{top_score*100:.1f}%',
        'top_k_matches':       [(str(sid), round(sc, 4)) for sid, sc in ranked[:top_k]],
        'n_beats_analysed':    len(beats),
        'auth_mode':           'identification_1_to_N',
        'liveness_confirmed':  bool(len(r_peaks) > 0),  # real heartbeat detected
        'heartrate_bpm':       round(60 / (np.diff(r_peaks).mean() / fs), 1)
                               if len(r_peaks) > 1 else 0,
    }

# Demo
demo_ecg = processed_records[0]['signal'] if processed_records else \
           np.sin(2*np.pi*1.2*np.arange(5000)/500)  # synthetic

result = ecg_biometric_identify(demo_ecg, templates_store)
print("=== MedVerse — ECG Biometric Identity ===")
for k, v in result.items():
    print(f"  {k:25s}: {v}")
